# Data Modules

> datasets designed to work specifically with VAE models.

In [ ]:
#| default_exp data.data_module

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os

import numpy as np
import pandas as pd
from omegaconf import OmegaConf
import torch
from torch.utils.data import DataLoader
import torchvision.transforms.v2 as v2
import torch.nn.functional as F

from c3jepa_wm.data.datasets import MultiAgentPOVDataset, MultiAgentWorldModelDataset, MultiAgentPlanningDataset
from c3jepa_wm.data.transforms import get_transforms

## Data Modules

In [ ]:
#| export
class DataModule:
    def __init__(self,
                 data_dir: str, 
                 batch_size: int = 64, 
                 shuffle: bool = True, 
                 num_workers: int = 0, 
                 pin_memory: bool = False,
                 drop_last: bool = False,
                 persistent_workers: bool = False):
        
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_workers = num_workers
        self.pin_memory = pin_memory
        self.drop_last = drop_last
        self.persistent_workers = persistent_workers

    def setup(self):
        """Load datasets and apply transforms."""
        raise NotImplementedError("Subclasses should implement this method to load datasets.")

    def get_data_path(self):
        paths = {
            "kaggle": "/kaggle/input/datasets/ahmedkhaledsal/findgoal/",
            "puhti": "/scratch/project_2009050/datasets/findgoal",
            "mahti": "/scratch/project_2009050/datasets/findgoal/",
            "fully_local": "/home/ahmed/Ahmed-home/1- Projects/Research/Journal 2/code/c3jepa-wm/0_results",
            "local": "/home/ahmed/Ahmed-home/1- Projects/Research/Journal 2/code/c3jepa-wm/mains/data",
        }

        for hostname, path in paths.items():
            if os.path.exists(path):
                print(f"Data path found for hostname: {hostname}")
                return path
        raise FileNotFoundError("No valid data path found for this hostname.")


    def train_dataloader(self):
        """Return the training dataloader."""
        return DataLoader(
            self.train_dataset, 
            batch_size=self.batch_size, 
            shuffle=self.shuffle, 
            num_workers=self.num_workers, 
            pin_memory=self.pin_memory,
            sampler=self.sampler(self.train_dataset),
            collate_fn=self.collator(),
            drop_last=self.drop_last,
            persistent_workers=self.persistent_workers
        )
    
    def val_dataloader(self):
        """Return the validation dataloader."""
        return DataLoader(
            self.val_dataset, 
            batch_size=self.batch_size, 
            shuffle=False, 
            num_workers=self.num_workers, 
            pin_memory=self.pin_memory,
            sampler=self.sampler(self.val_dataset),
            collate_fn=self.collator(),
            drop_last=self.drop_last,
            persistent_workers=self.persistent_workers
        )
    
    def test_dataloader(self):
        """Return the test dataloader (optional)."""
        return None

    def sampler(self, dataset):
        """Return a sampler for the given dataset (optional)."""
        dist_sampler = torch.utils.data.distributed.DistributedSampler(dataset) if torch.distributed.is_initialized() else None
        return dist_sampler
    

    def collator(self):
        """Custom collate function to handle variable-length sequences (optional)."""
        colate_fn= torch.utils.data.default_collate if torch.distributed.is_initialized() else None
        return colate_fn
    

In [ ]:
#| export
class VQDataModule(DataModule):
    def __init__(self,
                 batch_size: int = 64, 
                 img_size: int = 224,
                 **kwargs):
        
        data_dir = self.get_data_path()
        self.img_size = img_size
        super().__init__(data_dir= data_dir, batch_size= batch_size, **kwargs)
        
    def setup(self):
        self.train_transforms, self.val_transforms = get_transforms(img_size= self.img_size)
        
        self.train_dataset  = MultiAgentPOVDataset(
            h5_path= os.path.join(self.data_dir, "dataset.h5"), split= "train", transform= self.train_transforms
        )

        self.val_dataset  = MultiAgentPOVDataset(
            h5_path= os.path.join(self.data_dir, "dataset.h5"), split= "val", transform= self.val_transforms
        )
    


In [ ]:
#| export
class WMDataModule(DataModule):
    def __init__(self,
                 batch_size: int = 64, 
                 img_size: int = 224,
                 history_size: int = 3,
                 **kwargs):
        
        data_dir = self.get_data_path()
        self.img_size = img_size
        self.history_size = history_size
        super().__init__(data_dir= data_dir, batch_size= batch_size, **kwargs)
        

    def setup(self):

        self.train_sender_transform, self.val_sender_transform = get_transforms(img_size= self.img_size)
        self.train_receiver_transform, self.val_receiver_transform = get_transforms(img_size= self.img_size, model= "JEPA")
        
        self.train_dataset  = MultiAgentWorldModelDataset(
            h5_path= os.path.join(self.data_dir, "dataset.h5"), 
            split= "train", 
            history_size= self.history_size,
            receiver_transform= self.train_receiver_transform,
            sender_transform= self.train_sender_transform
        )

        self.val_dataset  = MultiAgentWorldModelDataset(
            h5_path= os.path.join(self.data_dir, "dataset.h5"), 
            split= "val",
            history_size= self.history_size,
            receiver_transform= self.val_receiver_transform,
            sender_transform= self.val_sender_transform
        )
        

In [ ]:
#| export
def planning_collate_fn(batch):
    """
    Collate variable-length episodes for MultiAgentPlanningDataset.

    Pads each agent's "pixels", "action", "pov_seq_vqvae", "csi" along the
    time dimension to the max length in the batch, and returns a "length"
    tensor + boolean "mask" (True = real timestep, False = padding) so
    downstream code can mask out padded steps.

    Input: list of dicts, each shaped like one MultiAgentPlanningDataset item
        {
            "agent_0": {"pixels": (T_i, C, H, W), "action": (T_i,),
                        "pov_seq_vqvae": (T_i, C, H, W), "csi": (T_i,)},
            "agent_1": {...},
            "episode_key": str,
            "length": int,
        }

    Output:
        {
            "agent_0": {
                "pixels": (B, T_max, C, H, W),
                "action": (B, T_max),
                "pov_seq_vqvae": (B, T_max, C, H, W),
                "csi": (B, T_max),
                "mask": (B, T_max) bool,
            },
            "agent_1": {...},
            "episode_key": list[str],
            "length": (B,) long,
        }
    """
    agent_keys = [k for k in batch[0].keys() if k.startswith("agent_")]
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    T_max = lengths.max().item()
    B = len(batch)

    out = {
        "episode_key": [item["episode_key"] for item in batch],
        "length": lengths,
    }

    for agent_key in agent_keys:
        sample = batch[0][agent_key]

        pixels = _pad_stack([item[agent_key]["pixels"] for item in batch], T_max)
        action = _pad_stack([item[agent_key]["action"] for item in batch], T_max)
        pov_vqvae = _pad_stack([item[agent_key]["pov_seq_vqvae"] for item in batch], T_max)
        csi = _pad_stack([item[agent_key]["csi"] for item in batch], T_max)

        mask = torch.arange(T_max).unsqueeze(0) < lengths.unsqueeze(1)  # (B, T_max)

        out[agent_key] = {
            "pixels": pixels,
            "action": action,
            "pov_seq_vqvae": pov_vqvae,
            "csi": csi,
            "mask": mask,
        }

    return out


def _pad_stack(tensors, T_max):
    """Pad a list of (T_i, ...) tensors along dim 0 to T_max and stack -> (B, T_max, ...)."""
    padded = []
    for t in tensors:
        pad_len = T_max - t.size(0)
        if pad_len > 0:
            pad_shape = (pad_len, *t.shape[1:])
            pad = torch.zeros(pad_shape, dtype=t.dtype)
            t = torch.cat([t, pad], dim=0)
        padded.append(t)
    return torch.stack(padded, dim=0)

In [ ]:
#| export
class PlanningDataModule(DataModule):
    def __init__(self,
                 batch_size: int = 64, 
                 img_size: int = 224,
                 max_length= 150,
                 **kwargs):
        
        data_dir = self.get_data_path()
        self.img_size = img_size
        self.max_length = max_length
        super().__init__(data_dir= data_dir, batch_size= batch_size, **kwargs)

    def setup(self):

        self.train_sender_transform, self.val_sender_transform = get_transforms(img_size= self.img_size)
        self.train_receiver_transform, self.val_receiver_transform = get_transforms(img_size= self.img_size, model= "JEPA")
        
        self.train_dataset  = MultiAgentPlanningDataset(
            h5_path= os.path.join(self.data_dir, "dataset.h5"), 
            split= "train", 
            jepa_transform= self.train_receiver_transform,
            vqvae_transform= self.train_sender_transform,
            max_length= self.max_length
        )

        self.val_dataset  = MultiAgentPlanningDataset(
            h5_path= os.path.join(self.data_dir, "dataset.h5"), 
            split= "val",
            jepa_transform= self.val_receiver_transform,
            vqvae_transform= self.val_sender_transform,
            max_length= self.max_length
        )

    def collator(self):
        """Custom collate function to handle variable-length episodes."""
        return planning_collate_fn
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()